# Model Experiments: Kryptonite-20, -30 and -40

The paper makes three claims about Kryptonite-n that this notebook tests:

1. Polynomial logistic regression stays below 59% up to degree 10, so polynomial basis expansion fails ("a counterexample to universal approximation").
2. Quadrupling the training data does not help.
3. Nothing reaches the 90% that counts as solving the dataset: even KryptoNet-40, an MLP pretrained on 200,000 examples of a related task and used frozen, scores worse than a coin flip.

Its Future Works suggests networks trained from scratch and tree-based models, and asks what degree and how many examples a polynomial would need.

KryptoNet-40 on Kryptonite-40-Adapt is the LoRA notebook's part. This notebook also picks one final pipeline per dataset, scores it on a locked test set and writes the hidden-set predictions to `hiddenlabels/`.

The helper code is in `experiment_tools.py` (needs `codecarbon`). Results are cached in `exp_cache/` (git-ignored) so an interrupted run resumes; delete it for a fresh run. `KRYP_QUICK=1` runs a dry run on 3,000 rows whose numbers mean nothing.

In [ ]:
import experiment_tools as et
from experiment_tools import G, NS, cv_by_fold, cv_config

print(
    "QUICK dry run" if et.QUICK else "Full run",
    f"- {et.K}-fold CV on 80% of the rows, 20% locked test",
)

## Data and Protocol

A stratified 20% of the labelled rows is locked away until the end. The other 80% (dev) is split into five stratified folds; every choice, including which bits to use, is made on the training folds only. Accuracies are mean ± sd over the folds; two models are compared fold by fold. The locked-test accuracy has the binomial 95% interval p ± 1.96·√(p(1−p)/N).

The inputs are ±1 bits plus Gaussian noise, so every model uses the sign of each input, except the paper's baseline, which uses the raw inputs as in the paper.

In [ ]:
et.check_protocol()

## Experiment 1: Higher Degree on All Bits

*Claim 1.* *Hypothesis:* a higher degree does not help on all n bits, because the number of monomials grows far faster than the number of training rows.

Logistic regression on explicit monomials (products of distinct bits), L1 or L2, `C` chosen by CV, up to the largest degree that fits in memory (4, 3, 2 for n = 20, 30, 40). Beyond that, the paper's own polynomial kernel (`C = 0.85`, plus an intercept) at degrees 6, 10, 15, n and 2n. From degree n on, the kernel can represent every function of the bits, so any failure there is not a lack of degree.

In [ ]:
et.test_kernel()
for n in NS:
    cv_config(n, "BASE", "paper", "all", "deg2", dict(deg=2, C=0.85))
    for d in range(1, G["MAX_DEG"][n] + 1):
        for pen in ("l2", "l1"):
            for C in G["C_L2" if pen == "l2" else "C_L1"]:
                cv_config(n, "LR", f"lr_{pen}", "all", f"deg{d}", dict(deg=d, C=C))
    for d in G["KP_DEGS"][n]:
        cv_config(n, "KPOLY", "kpoly", "all", f"deg{d}", dict(deg=d, C=G["KP_C"]))
et.table_degree_all()

## Experiment 2: Models Trained From Scratch on All Bits

*Claim 3.* The two model types the paper's Future Works suggests, tuned by the same CV on all n bits:

- a random forest;
- an MLP with KryptoNet-40's architecture (four hidden layers of 512), trained from scratch on Kryptonite-n with Adam and early stopping. The paper only used this architecture pretrained on Kryptonite-40-Pre and frozen.

In [ ]:
for n in NS:
    for f in G["RF_MAXF"]:
        for leaf in G["RF_LEAF"]:
            cv_config(n, "RF", "rf", "all", "rf", dict(max_features=f, msl=leaf))
    if et.RUN_MLP:
        cv_by_fold(n, "mlp", dict(h=G["MLP_H"]), "4x512")
et.table_families()

## Experiment 3: Which Bits Matter?

The forest beats every polynomial on the same bits, and a forest only splits on bits that help it. *Hypothesis:* the label depends on a few bits, and the polynomials fail because almost all their monomials involve the others.

We ask the forest which bits it relies on with recursive feature elimination (Guyon et al., 2002), inside each training fold:

1. Hold out a quarter of the training rows. Fit a forest on the rest and record its accuracy on that quarter.
2. Remove the bit with the lowest permutation importance (Breiman, 2001): the drop in that accuracy when the bit is shuffled. (The forest's built-in impurity importance is biased; Strobl et al., 2007.)
3. Repeat until one bit is left, then keep the smallest set within one standard error of the best (at most 14 bits, so every polynomial on them fits in memory).

If the hypothesis holds, accuracy rises as bits are removed and drops sharply at the first relevant one. The elimination is repeated with two more forest seeds to check that it is stable.

In [ ]:
et.table_elimination()
et.fig_elimination()

## Experiment 4: Degree on the Kept Bits

*Claim 1, and the paper's question of what degree is needed.* The same model as Experiment 1, logistic regression on all monomials up to degree d, on the bits kept inside each training fold.

In [ ]:
for n in NS:
    for d in G["SEL_DEGS"]:
        for C in G["C_L2"]:
            cv_config(n, "LR", "lr_l2", "sel", f"deg{d}", dict(deg=d, C=C))
et.table_degree_sel()
et.fig_degree()

## Experiment 5: Amount of Training Data

*Claim 2, and the paper's question of how many examples are needed.* The best model of each kind is trained on nested random subsets of the training folds (first three folds) and scored on the held-out fold. For LR, `C` is scaled with the number of rows so the penalty per row stays fixed.

In [ ]:
et.fig_learning_curves(et.run_learning_curves())

## Final Pipeline and Hidden-Set Predictions

The final pipeline per n is the candidate with the highest CV mean. It is refitted on all dev rows (bit elimination included) and scored once on the locked rows. Two controls must score about 0.5: shuffled training labels (otherwise something leaks) and the same model on as many eliminated bits (otherwise the kept bits are not what matters). `check_caps` lists any fit that stopped at its iteration limit.

For the hidden set the pipeline is refitted on all labelled rows. As the hidden labels are unknown, its predictions must agree with the MLP's: two models with error rates e₁ and e₂ can disagree on at most e₁ + e₂ of the rows, so a lower agreement (with 2 points of slack) stops the notebook. Then `hiddenlabels/y_predicted_{20,30,40}.npy` are written and checked with the course's `check_submission.py`.

In [ ]:
FINAL = et.choose_final()
et.run_controls(FINAL)
et.table_final(FINAL)
et.check_caps()
print()
et.predict_hidden(FINAL)

## Summary

In [ ]:
et.fig_summary_table(FINAL)
et.table_summary(FINAL)

## Energy and Emissions

Measured with CodeCarbon while each result was computed: CPU energy from the processor's own counters (machine-wide, so other programs should be closed), and the GPU's idle draw, since no model here uses it. RAM and screen are not measured; if the CPU counters cannot be read, nothing is reported. Emissions use the GB grid's 2024 average, 125 g CO2 per kWh (NESO, *Britain's Electricity Explained: 2024 Review*).

In [ ]:
et.carbon_report()

## Findings

- **Claim 1 (polynomials) is wrong about the cause.** On all bits more degree does not help: explicit monomials reach at most 0.68, and the kernel scores 0.50–0.60 at every degree from 6 to 2n while fitting its training rows perfectly. But the label depends on only 10 bits (the same 10 in every fold and with every forest seed), and the same polynomial on those bits scores 0.96 from degree 6. Degree 5 scores 0.78–0.79. On 10 eliminated bits it scores 0.50–0.51.
- **Claim 2 (data) is wrong for a model that can use the data.** The kept-bits polynomial rises from near chance at 250–500 rows to 0.89–0.94 at 4,000 and 0.96 with all rows (7,680–20,480).
- **Claim 3 (nothing reaches 90%) is wrong.** KryptoNet-40's architecture trained from scratch on all bits scores 0.931 / 0.943 / 0.949 without any bit selection. It needs the data, though: with 4,000 rows it is still at 0.61 or below, which agrees with the paper's caution about training it on a few thousand examples.
- **Final pipeline:** the kept-bits polynomial. Locked test 0.958 / 0.959 / 0.959; it beats the MLP on 5 of 5 folds for every n (by 0.027 / 0.019 / 0.011), needs far less data and names the bits that matter.